In [ ]:
pip install --upgrade "mlflow>=3.1"


^C
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: C:\Users\timur\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# Imports

import mlflow
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# The word2vec imports

import gensim
import gensim.downloader as api
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# The LSTM imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import TransformerEncoder, TransformerEncoderLayer

In [3]:
os.chdir(r"C:\Users\timur\Documents\GitHub\EmailSentin\backend_docker\mlflow")

In [4]:
mlflow.set_tracking_uri(r"sqlite:///C:\Users\timur\Documents\GitHub\EmailSentin\backend_docker\mlflow\mlflow.db")
mlflow.set_experiment("my-first-experiment")

2026/02/08 21:10:36 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/02/08 21:10:36 INFO mlflow.store.db.utils: Updating database tables
2026/02/08 21:10:36 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/08 21:10:36 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/02/08 21:10:37 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/08 21:10:37 INFO alembic.runtime.migration: Will assume non-transactional DDL.


<Experiment: artifact_location='file:///c:/Users/timur/AppData/Local/Programs/Microsoft VS Code/mlruns/1', creation_time=1768519295232, experiment_id='1', last_update_time=1768519295232, lifecycle_stage='active', name='my-first-experiment', tags={}>

In [6]:
# Loading the model


import torch
import torch.nn as nn

class BiLSTMWithAttention(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, output_dim):
        super(BiLSTMWithAttention, self).__init__()
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.attention = nn.Linear(hidden_dim * 2, 1)  # Bidirectional -> 2 * hidden_dim
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
     lstm_out, _ = self.lstm(x)  # Shape: (batch_size, seq_len, hidden_dim * 2)

     # Calculate attention scores
     attn_weights = torch.softmax(self.attention(lstm_out), dim=1)  # Shape: (batch_size, seq_len, 1)
    
     # Calculate the context vector as a weighted sum of the LSTM outputs
     context_vector = torch.sum(attn_weights * lstm_out, dim=1)  # Shape: (batch_size, hidden_dim * 2)

     # Fully connected layer for classification
     logit = self.fc(context_vector)

     # Apply softmax to the output to get probabilities
     output = torch.sigmoid(logit)
     
    
     return output,logit,attn_weights.squeeze(-1)  # Return the attention weights


# Initialize the model
model = api.load('glove-twitter-25')
modelLSTM = BiLSTMWithAttention(embedding_dim=model.vector_size, hidden_dim=128, output_dim=1)  # Adjust parameters as needed


state_dict_path = r'C:\Users\timur\Documents\GitHub\EmailSentin\backend_docker\lstm_based\models\model1\model_2_atn.pth'
state_dict = torch.load(
    state_dict_path,
    map_location="cpu"
)
modelLSTM.load_state_dict(state_dict)

<All keys matched successfully>

In [7]:

def sentences_to_vectors(sentences, model):
    encoding_dim = model.vector_size
    no_sentence = len(sentences)
    max_word_count = 50
    
    # Initialize a 3D array with zeros
    returned_array = np.zeros((encoding_dim, max_word_count, no_sentence))

    def tokenize(sentence):
        tokens = word_tokenize(sentence)  # Tokenize sentence
        return [word for word in tokens if word not in stopwords.words('english')]  # Remove stopwords

    for i, sentence in enumerate(sentences):
        words = tokenize(sentence)
        word_vectors = [model[word] for word in words if word in model]
        
        # Pad or truncate word_vectors to fit max_word_count
        if len(word_vectors) < max_word_count:
            # Pad with zeros if there are fewer than max_word_count word vectors
            padded_vectors = np.array(word_vectors + [[0] * encoding_dim] * (max_word_count - len(word_vectors)))
        else:
            # Truncate if there are more than max_word_count word vectors
            padded_vectors = np.array(word_vectors[:max_word_count])
        
        # Fill the 3D array
        returned_array[:, :, i] = padded_vectors.T  # Transpose to match shape (encoding_dim, max_word_count)

    return returned_array


def convert_column_sentences_to_vectors(df, column_name, model):
    encoding_dim = model.vector_size
    no_sentence = len(df[column_name])
    max_word_count = 100
    
    # Initialize a 3D array with zeros
    returned_array = np.zeros((encoding_dim, max_word_count, no_sentence))

    meaningless_words = {}  

    def tokenize(sentence):
        tokens = word_tokenize(sentence)  # Tokenize sentence
        return [word for word in tokens if word.lower() not in stopwords.words('english') and word.lower() not in meaningless_words]  # Remove stopwords and meaningless words

    for i, sentence in enumerate(df[column_name]):
        words = tokenize(sentence)
        word_vectors = [model[word] for word in words if word in model]
        
        # Pad or truncate word_vectors to fit max_word_count
        if len(word_vectors) < max_word_count:
            # Pad with zeros if there are fewer than max_word_count word vectors
            padded_vectors = np.array(word_vectors + [[0] * encoding_dim] * (max_word_count - len(word_vectors)))
        else:
            # Truncate if there are more than max_word_count word vectors
            padded_vectors = np.array(word_vectors[:max_word_count])
        
        # Fill the 3D array
        returned_array[:, :, i] = padded_vectors.T  # Transpose to match shape (encoding_dim, max_word_count)

    return returned_array

In [20]:
import mlflow
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch

# Assuming you have these functions/variables defined elsewhere
# from your_model import convert_column_sentences_to_vectors
# model = ...  # Your model for encoding

data_dir = r"C:\Users\timur\Documents\GitHub\EmailSentin\data\ready_data.csv"
df_processed_data = pd.read_csv(data_dir)

mapping = {
    'Very Informal': -2,
    'Informal': -1,
    'Neutral': 0,
    'Formal': 1,
    'Very Formal': 2
}

# Encode the sentences
encoded_matrix = convert_column_sentences_to_vectors(df_processed_data, 'text', model)
df_processed_data['numerical_label'] = df_processed_data['label'].map(mapping)
ground_truth = df_processed_data['numerical_label']

# Prepare the data
encoded_matrix_transposed = np.transpose(encoded_matrix, (2, 0, 1))  # Rearrange to shape (1333, 25, 25)

# Perform train-test split
X_train, X_test, y_train, y_test = train_test_split(
    encoded_matrix_transposed, 
    ground_truth, 
    test_size=0.2,
    random_state=41,
    shuffle=True
)

# Process training data
y_train = np.array(y_train)
y_train = np.nan_to_num(y_train, nan=0.0)
train_mask = y_train != 0
X_train_filtered = X_train[train_mask]
y_train_filtered = y_train[train_mask]

y_train_binary = np.where(y_train_filtered > -1, 1, 0)
y_train_binary = torch.tensor(y_train_binary, dtype=torch.float32)
X_train_tensor = torch.tensor(X_train_filtered, dtype=torch.float32).transpose(-1, -2)

# Process test data
y_test = np.array(y_test)
test_mask = y_test != 0
X_test_filtered = X_test[test_mask]
y_test_filtered = y_test[test_mask]

y_test_binary = np.where(y_test_filtered > 0, 1, 0)  # Note: Fixed typo here (was using y_train > 0)
y_test_binary = torch.tensor(y_test_binary, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_filtered, dtype=torch.float32).transpose(-1, -2)

# Create DataFrames for MLflow datasets
train_df = pd.DataFrame({
    'features': list(X_train_tensor.numpy()),  # Convert tensors to numpy for storage
    'target': y_train_binary.numpy()
})

test_df = pd.DataFrame({
    'features': list(X_test_tensor.numpy()),
    'target': y_test_binary.numpy()
})

# Create MLflow dataset objects
train_dataset = mlflow.data.from_pandas(
    train_df,
    source=data_dir,
    name="email-formality-train",
    targets="target"
)

test_dataset = mlflow.data.from_pandas(
    test_df,
    source=data_dir,
    name="email-formality-test",
    targets="target"
)


# You can also log dataset information
print(f"Train dataset: {train_dataset}")
print(f"Test dataset: {test_dataset}")




Train dataset: <mlflow.data.pandas_dataset.PandasDataset object at 0x0000017001997690>
Test dataset: <mlflow.data.pandas_dataset.PandasDataset object at 0x0000017001BD3AD0>


C:\Users\timur\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: Failed to determine whether UCVolumeDatasetSource can resolve source information for 'C:\Users\timur\Documents\GitHub\EmailSentin\data\ready_data.csv'. Exception: 
  return _dataset_source_registry.resolve(
C:\Users\timur\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
C:\Users\timur\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\mlflow\data\dataset_source_r

In [29]:
import torch
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score



modelLSTM.eval()  # Set the model to evaluation mode

# Initialize variables to keep track of correct predictions and true labels
correct = 0
total = 0
all_preds = []
all_targets = []
batch_size  =32
# No need to calculate gradients during testing
with torch.no_grad():
    for i in range(0, X_test_tensor.size(0), batch_size):
        # Get the current test batch
        batch_data = X_test_tensor[i:i + batch_size]
        batch_targets = y_test_binary[i:i + batch_size]

        # Reshape the batch targets to match the shape of outputs
        batch_targets = batch_targets.unsqueeze(1)  # Make it (batch_size, 1)

        # Forward pass to get predictions
        outputs,_,_= modelLSTM(batch_data)


        # Apply sigmoid to get probabilities, and convert to binary predictions (0 or 1)
        
        predicted = (outputs > 0.5).float()  # Threshold at 0.41 for binary classification
        

        # Store predictions and true labels for confusion matrix
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(batch_targets.cpu().numpy())

        # Update the count of correct predictions
        total += batch_targets.size(0)  # Total number of samples in this batch
        correct += (predicted == batch_targets).sum().item()  # Count correct predictions

# Calculate accuracy
y_true = np.array(all_targets)
y_pred = np.array(all_preds)

# Compute metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Classification metrics dictionary
classification_metrics = {
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1_score": f1,
    "true_positives": int(cm[1, 1]),
    "true_negatives": int(cm[0, 0]),
    "false_positives": int(cm[0, 1]),
    "false_negatives": int(cm[1, 0])
}

print(classification_metrics)


{'accuracy': 0.8838709677419355, 'precision': 0.8970588235294118, 'recall': 0.8472222222222222, 'f1_score': 0.8714285714285713, 'true_positives': 61, 'true_negatives': 76, 'false_positives': 7, 'false_negatives': 11}


In [30]:
modelLSTM

BiLSTMWithAttention(
  (lstm): LSTM(25, 128, batch_first=True, bidirectional=True)
  (attention): Linear(in_features=256, out_features=1, bias=True)
  (fc): Linear(in_features=256, out_features=1, bias=True)
)

In [43]:
df_train = pd.DataFrame(X_train)  # features
df_train['label'] = y_train       # append label column

df_test = pd.DataFrame(X_test)
df_test['label'] = y_test


print(df_train.head())
print(df_test.head())

ValueError: Must pass 2-d input. shape=(635, 100, 25)

In [48]:
model_code = """
import torch
import torch.nn as nn

class BiLSTMWithAttention(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, output_dim):
        super(BiLSTMWithAttention, self).__init__()
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.attention = nn.Linear(hidden_dim * 2, 1)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attn_weights = torch.softmax(self.attention(lstm_out), dim=1)
        context_vector = torch.sum(attn_weights * lstm_out, dim=1)
        logit = self.fc(context_vector)
        output = torch.sigmoid(logit)
        return output, logit, attn_weights.squeeze(-1)
"""
with open("BiLSTMWithAttention.py", "w") as f:
    f.write(model_code)

In [34]:
np.save("X_train.npy", X_train)
np.save("X_test.npy", X_test)
np.save("y_train.npy", y_train)
np.save("y_test.npy", y_test)
with mlflow.start_run() as run:
    hyperparams = {
    "embedding_model_name": "glove-twitter-25",  # embedding info
    "embedding_dim": model.vector_size,          # vector size from the loaded model
    "hidden_dim": modelLSTM.lstm.hidden_size,
    "bidirectional": modelLSTM.lstm.bidirectional,
    "output_dim": modelLSTM.fc.out_features,
    "learning_rate": 0.001,
    "batch_size": 32

} 
    mlflow.log_params(hyperparams)
    
    mlflow.log_metrics(classification_metrics)
    mlflow.log_input(train_dataset, context="training")
    mlflow.log_input(test_dataset, context="testing")

    # mlflow.log_artifact("X_train.npy", artifact_path="datasets/train")
    # mlflow.log_artifact("X_test.npy", artifact_path="datasets/validation")
    # mlflow.log_artifact("y_train.npy", artifact_path="datasets/train")
    # mlflow.log_artifact("y_test.npy", artifact_path="datasets/validation")
    mlflow.log_artifact("BiLSTMWithAttention.py", artifact_path="model_code")
    mlflow.pytorch.log_model(
            pytorch_model=modelLSTM,
            name="pytorch_model_1",
            registered_model_name="email_formality_pytorch_main"  # Optional: register in model registry
        )
    active_run = mlflow.active_run()
    print("Run info:")
    print(f"Run ID: {active_run.info.run_id}")
    print(f"Experiment ID: {active_run.info.experiment_id}")
    print(f"Status: {active_run.info.status}")
    print(f"Artifact URI: {active_run.info.artifact_uri}")
    

Run info:
Run ID: 9c78e5dac11d4b479e3943350192354e
Experiment ID: 1
Status: RUNNING
Artifact URI: file:///c:/Users/timur/AppData/Local/Programs/Microsoft VS Code/mlruns/1/9c78e5dac11d4b479e3943350192354e/artifacts


Registered model 'email_formality_pytorch_main' already exists. Creating a new version of this model...
Created version '2' of model 'email_formality_pytorch_main'.


SyntaxError: incomplete input (1963159769.py, line 103)

In [2]:
import subprocess
os.chdir(r'C:\Users\timur\Documents\GitHub\EmailSentin')
git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode("utf-8").strip()


In [3]:
git_commit

'b2ace024db1dedd969ae6933667fb6a96a1bc864'

In [36]:
# migration_plan.py
class LambdaToSageMakerMigration:
    def __init__(self):
        self.metrics = {
            "current_lambda": {},
            "projected_sagemaker": {}
        }
    
    def analyze_current_performance(self):
        """Analyze if Lambda is sufficient"""
        
        issues = []
        
        # Check Lambda limits
        if self.metrics["current_lambda"]["avg_duration"] > 10000:  # >10 seconds
            issues.append("❌ Lambda timeout risk (>10s)")
        
        if self.metrics["current_lambda"]["memory_usage"] > 8000:  # >8GB
            issues.append("❌ Approaching Lambda memory limit")
        
        if self.metrics["current_lambda"]["cold_start_rate"] > 0.1:  # >10%
            issues.append("❌ High cold start rate")
        
        if self.metrics["current_lambda"]["concurrent_executions"] > 900:
            issues.append("❌ Approaching Lambda concurrency limit")
        
        return issues
    
    def migration_decision_tree(self):
        """Decision tree for Lambda vs SageMaker"""
        
        print("=== Should you migrate from Lambda to SageMaker? ===\n")
        
        questions = [
            ("Average request duration > 5 seconds?", "SageMaker"),
            ("Memory requirement > 8GB?", "SageMaker"),
            ("Cold starts affecting user experience?", "SageMaker"),
            ("Need persistent WebSocket connections?", "SageMaker"),
            ("Need GPU acceleration?", "SageMaker"),
            ("Traffic < 100k requests/month?", "Lambda"),
            ("Cost-sensitive with sporadic traffic?", "Lambda"),
            ("Already have Lambda container working?", "Keep Lambda + improve")
        ]
        
        sagemaker_points = 0
        lambda_points = 0
        
        for question, recommendation in questions:
            answer = input(f"{question} (y/n): ").lower()
            if answer == 'y':
                if recommendation == "SageMaker":
                    sagemaker_points += 1
                else:
                    lambda_points += 1
        
        print(f"\nResults: SageMaker={sagemaker_points}, Lambda={lambda_points}")
        
        if sagemaker_points > lambda_points:
            print("✅ Recommendation: Migrate to SageMaker")
            return "sagemaker"
        elif lambda_points > sagemaker_points:
            print("✅ Recommendation: Stay with Lambda (optimize)")
            return "lambda"
        else:
            print("⚖️  Tie: Consider hybrid approach")
            return "hybrid"
    
    def hybrid_approach(self):
        """Hybrid: Lambda for API, SageMaker for heavy lifting"""
        
        return {
            "architecture": """
                API Gateway → Lambda (orchestrator) → SageMaker (inference)
                         ↳ Fallback to Lambda if SageMaker busy
            """,
            "benefits": [
                "Cost-effective for low traffic",
                "Scalable for peaks",
                "No cold starts for critical requests"
            ],
            "implementation": """
                1. Keep existing Lambda API
                2. Add SageMaker endpoint for heavy requests
                3. Smart router in Lambda decides where to send
                4. Monitor and adjust routing logic
            """
        }

In [38]:

param_dict = {name: param for name, param in modelLSTM.named_parameters()}

In [40]:
!mlflow server --port 5000

^C


In [5]:
os.chdir(r"C:\Users\timur\Documents\GitHub\EmailSentin\backend_docker\mlflow")
!mlflow server --port 5000

^C
